# One XBRL Concept Over Time

Pull a single financial line item across reporting periods with the **company concept** endpoint, then tidy it into a `NamedTuple` table you can hand to TimeSeries or any Tables.jl sink.

In [ ]:
# import Pkg; Pkg.add(url = "https://github.com/Trumpingtons/EDGAR.jl")
using EDGAR

# Set your contact once via "Set contact" on the webview; it is injected as
# ENV["SEC_USER_AGENT"], which EDGAR.jl reads automatically. Or set it explicitly:
# set_user_agent("Your Name your.email@example.com")

## Fetch Net Income

`/api/xbrl/companyconcept/CIK#/us-gaap/<Tag>.json` returns every value reported for one `us-gaap` concept, grouped by unit.

In [ ]:
concept = company_concept("320193", "us-gaap", "NetIncomeLoss")     # Apple

println("Units reported: ", collect(keys(concept.units)))

## Inspect an Observation

Each entry under a unit carries the period (`start`/`end`), the value, the fiscal year/period, and which `form` it came from (10-K = annual, 10-Q = quarterly).

In [ ]:
o = last(concept.units.USD)
(; o.end, o.fy, o.fp, o.form, o.val)

## Tidy Into a Table

Keep the annual (10-K) figures and build a `Vector` of `NamedTuple`s — a Tables.jl source, so it pipes into the container of your choice.

In [ ]:
rows = [(; fy = o.fy, period_end = String(o.end), value = o.val)
        for o in concept.units.USD if o.form == "10-K"]

for r in last(rows, 6)
    println(r.fy, "  ", r.period_end, "  ", r.value)
end

# using Tables;     Tables.columntable(rows)
# using TimeSeries;  TimeArray(Date.(getfield.(rows, :period_end)), getfield.(rows, :value), [:NetIncome])

## Summary
- The company-concept endpoint narrows the data to one `us-gaap` tag, grouped by unit.
- Each observation has `start`/`end`, `val`, `fy`/`fp`, and `form`; filter on `form == "10-K"` for annual figures.
- A `Vector` of `NamedTuple`s is a Tables.jl source — pipe it into TimeSeries or any other Tables.jl sink.